# Workflow - Prompt Chaining

## Load Keys to Env.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
from cohere import ClientV2

co = ClientV2(api_key=os.environ["COHERE_API_KEY"])

MODEL_NAME = "command-a-03-2025"

## Helper for LLM Calls

In [4]:
def llm(prompt: str, system: str = "") -> str:
    msgs = []
    if system:
        msgs.append(
            {
                "role": "system",
                "content": system,
            }
        )
    msgs.append(
        {
            "role": "user",
            "content": prompt,
        }
    )

    return (
        co.chat(
            model=MODEL_NAME,
            messages=msgs,
        )
        .message.content[0]
        .text.strip()
    )

## Define the Stages in the Prompt Chain

In [6]:
product = (
    "NoiseNest is a $129 portable white-noise machine with 30 ambient sounds, "
    "a 40-hour battery, an app-free design, and a child-safe lock."
)

# Step 1: Extract the features
features = llm(
    f"Extract the three most marketable features from this product:\n\n{product}\n\n" ""
    "Return a short bulleted list"
)

# Step 2: Create Tagline - Uses Features from Step 1
tagline = llm(
    f"Write a single punchy tagline(max 10 words) for a product with these features:\n\n{features}"
)


# Step 3: Create Social Media Post - Uses Tagline from Step 2
post = llm(
    f"Expand this tagline into a 2-sentence Instagram caption with one emoji:\n\nTagline: {tagline}"
)

print("-----Features----\n", features)
print("\n\n-----Tagline----\n", tagline)
print("\n\n-----Social Post----\n", post)

-----Features----
 Here are the three most marketable features of NoiseNest:

* **40-hour battery life:** Long-lasting battery is a huge selling point for portability and convenience.

* **30 ambient sounds:**  A wide variety of sounds caters to diverse preferences and needs.

* **Child-safe lock:**  This feature appeals to parents and caregivers, adding a layer of safety and peace of mind.


-----Tagline----
 "NoiseNest: 40-Hour Escape, 30 Sounds, Child-Safe Serenity."


-----Social Post----
 Escape the chaos and find your zen with NoiseNest—40 hours of uninterrupted bliss, 30 soothing sounds, and child-safe serenity for the whole family. 🌿 #NoiseNest #SerenityNow #FamilyFriendly


## Disadvantages of Using a Long Prompt for multiple tasks

1. The model may skip steps or merge them.
2. If the tagline is bad, you have no way to fix it without re-running the whole thing.
3. The output format is harder to control.

> A small but important upgrade: validation gates

> Real chains often add a tiny gate between steps — "is this output the right shape?" — and abort early if not. This catches problems closer to where they happen. We won't implement one here, but keep it in mind for your own pipelines.